In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings('ignore')


In [ ]:
class FinancialFeatureEngineer:
    """
    Comprehensive feature engineering for financial price prediction
    Binary classification: BUY if return > 0, SELL otherwise
    Includes all original ~120 features
    """
    
    def __init__(self, offset_days=20):
        self.offset_days = offset_days
        self.fitted_volatility_threshold = None
        
    def calculate_returns_and_moving_averages(self, df):
        """Calculate returns and moving averages"""
        df = df.copy()
        
        # Basic returns
        df['1d_return'] = df.groupby('Symbol')['Close'].pct_change()
        df['5d_ROC'] = df.groupby('Symbol')['Close'].pct_change(5)
        df['10d_ROC'] = df.groupby('Symbol')['Close'].pct_change(10)
        df['20d_ROC'] = df.groupby('Symbol')['Close'].pct_change(20)
        df['momentum_ratio'] = df.groupby('Symbol')['Close'].transform(lambda x: x / x.shift(5))
        
        # Simple Moving Averages
        df['SMA_10'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(10).mean())
        df['SMA_20'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(20).mean())
        df['SMA_50'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(50).mean())
        
        # Exponential Moving Averages
        df['EMA_12'] = df.groupby('Symbol')['Close'].transform(lambda x: x.ewm(span=12).mean())
        df['EMA_26'] = df.groupby('Symbol')['Close'].transform(lambda x: x.ewm(span=26).mean())
        df['ema_5'] = df.groupby('Symbol')['Close'].transform(lambda x: x.ewm(span=5).mean())
        df['ema_20'] = df.groupby('Symbol')['Close'].transform(lambda x: x.ewm(span=20).mean())
        
        # Price to MA ratios
        df['close_to_SMA_10'] = df['Close'] / df['SMA_10']
        df['close_to_SMA_20'] = df['Close'] / df['SMA_20']
        df['close_to_SMA_50'] = df['Close'] / df['SMA_50']
        df['ema_ratio'] = df['ema_5'] / df['ema_20']
        
        return df

    
    def calculate_momentum_indicators(self, df):
        """Calculate comprehensive momentum indicators"""
        # MACD
        df['MACD'] = df['EMA_12'] - df['EMA_26']
        df['MACD_signal'] = df.groupby('Symbol')['MACD'].transform(lambda x: x.ewm(span=9).mean())
        df['MACD_histogram'] = df['MACD'] - df['MACD_signal']
        
        # RSI
        def calculate_rsi(series, window=14):
            delta = series.diff()
            gain = delta.where(delta > 0, 0)
            loss = -delta.where(delta < 0, 0)
            
            avg_gain = gain.rolling(window=window).mean()
            avg_loss = loss.rolling(window=window).mean()
            
            rs = avg_gain / avg_loss
            rsi = 100 - (100 / (1 + rs))
            return rsi
        
        df['RSI_14'] = df.groupby('Symbol')['Close'].transform(calculate_rsi)
        
        
        
        def compute_williams_r(df):
            # If there's only one symbol, skip groupby to avoid DataFrame return
            if df['Symbol'].nunique() == 1:
                high14 = df['High'].rolling(14).max()
                low14 = df['Low'].rolling(14).min()
                wr = (high14 - df['Close']) / (high14 - low14) * -100
                return wr

            # Otherwise, handle groupwise
            return df.groupby('Symbol', group_keys=False).apply(
                lambda g: (g['High'].rolling(14).max() - g['Close']) /
                          (g['High'].rolling(14).max() - g['Low'].rolling(14).min()) * -100
            ).reset_index(level=0, drop=True)

        # then:
        df['williams_R_14'] = compute_williams_r(df)

        
        return df
    
    def calculate_volatility_features(self, df):
        """Calculate comprehensive volatility features"""
        # Volatility measures
        df['volatility_10'] = df.groupby('Symbol')['1d_return'].transform(lambda x: x.rolling(10).std())
        df['volatility_20'] = df.groupby('Symbol')['1d_return'].transform(lambda x: x.rolling(20).std())
        df['volatility_60'] = df.groupby('Symbol')['1d_return'].transform(lambda x: x.rolling(60).std())
        
        # Normalized range
        df['normalized_range'] = (df['High'] - df['Low']) / df['Close']
        df['rolling_std_5'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(5).std())
        df['rolling_std_20'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(20).std())
        df['volatility_ratio'] = df['rolling_std_5'] / df['rolling_std_20']
        
        # Average True Range (ATR)
        def calculate_atr(group):
            high_low = group['High'] - group['Low']
            high_close_prev = abs(group['High'] - group['Close'].shift(1))
            low_close_prev = abs(group['Low'] - group['Close'].shift(1))
            true_range = pd.concat([high_low, high_close_prev, low_close_prev], axis=1).max(axis=1)
            return true_range.rolling(14).mean()
        
        df['ATR_14'] = df.groupby('Symbol').apply(calculate_atr).reset_index(level=0, drop=True)
        df['true_range'] = df['High'] - df['Low']
        df['atr_14'] = df.groupby('Symbol')['true_range'].transform(lambda x: x.rolling(14).mean())
        
        # Bollinger Bands
        df['bollinger_upper'] = df['SMA_20'] + 2 * df['volatility_20']
        df['bollinger_lower'] = df['SMA_20'] - 2 * df['volatility_20']
        df['bollinger_position'] = (df['Close'] - df['bollinger_lower']) / (df['bollinger_upper'] - df['bollinger_lower'])
        
        # Sharpe ratio
        df['sharpe_20'] = df.groupby('Symbol')['1d_return'].transform(
            lambda x: x.rolling(20).mean() / x.rolling(20).std()
        )
        
        return df
    
    def calculate_volume_features(self, df):
        """Calculate comprehensive volume features"""
        # Volume features
        df['volume_SMA_20'] = df.groupby('Symbol')['Volume'].transform(lambda x: x.rolling(20).mean())
        df['volume_SMA_10'] = df.groupby('Symbol')['Volume'].transform(lambda x: x.rolling(10).mean())
        df['volume_SMA_5'] = df.groupby('Symbol')['Volume'].transform(lambda x: x.rolling(5).mean())
        
        df['volume_ratio'] = df['Volume'] / df['volume_SMA_20']
        df['vol_ma_5'] = df.groupby('Symbol')['Volume'].transform(lambda x: x.rolling(5).mean())
        df['vol_ma_20'] = df.groupby('Symbol')['Volume'].transform(lambda x: x.rolling(20).mean())
        df['vol_ratio'] = df['vol_ma_5'] / df['vol_ma_20']
        df['vwap_ratio'] = df['Close'] / df['VWAP']
        
        # On-Balance Volume (OBV)
        def calculate_obv(group):
            obv = (group['Volume'] * np.where(group['Close'] > group['Close'].shift(1), 1, 
                                           np.where(group['Close'] < group['Close'].shift(1), -1, 0))).cumsum()
            return obv
        
        df['OBV'] = df.groupby('Symbol').apply(calculate_obv).reset_index(level=0, drop=True)
        
        # Volume-return correlation
        def volume_return_corr(group):
            return group['1d_return'].rolling(20).corr(group['Volume'])
        
        df['volume_return_corr_20'] = df.groupby('Symbol').apply(volume_return_corr).reset_index(level=0, drop=True)
        
        return df
    
    def calculate_candlestick_features(self, df):
        """Calculate candlestick patterns and features"""
        df['candle_body'] = abs(df['Close'] - df['Open'])
        df['upper_shadow'] = df['High'] - df[['Close', 'Open']].max(axis=1)
        df['lower_shadow'] = df[['Close', 'Open']].min(axis=1) - df['Low']
        
        # Bullish engulfing pattern
        df['bullish_engulfing'] = ((df['Close'] > df['Open']) &
                                   (df['Close'].shift(1) < df['Open'].shift(1)) &
                                   (df['Close'] > df['Open'].shift(1)) &
                                   (df['Open'] < df['Close'].shift(1))).astype(int)
        
        return df
    
    def create_time_features(self, df):
        """Create comprehensive time-based features"""
        df['Date'] = pd.to_datetime(df['Date'])
        df['day_of_week'] = df['Date'].dt.dayofweek
        df['month'] = df['Date'].dt.month
        df['day_of_month'] = df['Date'].dt.day
        df['is_month_start'] = df['Date'].dt.is_month_start.astype(int)
        df['is_month_end'] = df['Date'].dt.is_month_end.astype(int)
        df['is_quarter_end'] = (df['Date'].dt.month % 3 == 0) & df['Date'].dt.is_month_end
        
        # One-hot encode categorical time features
        time_dummies = pd.get_dummies(df['day_of_week'], prefix='dow')
        month_dummies = pd.get_dummies(df['month'], prefix='month')
        
        df = pd.concat([df, time_dummies, month_dummies], axis=1)
        
        return df
    
    def create_tier1_raw_lags(self, df):
        """Create Tier 1: Raw lag features for critical indicators"""
        lag_features = {
            '1d_return': [1, 2, 3, 5],
            'volume_ratio': [1, 2, 5],
            'RSI_14': [1, 3, 5],
            'normalized_range': [1, 3],
            'vol_ratio': [1, 2, 3],
            'vwap_ratio': [1, 2, 3],
            'ema_ratio': [1, 2, 3],
            'MACD': [1, 2],
            'bollinger_position': [1, 2]
        }
        
        for feature, lags in lag_features.items():
            for lag in lags:
                df[f'{feature}_lag_{lag}'] = df.groupby('Symbol')[feature].shift(lag)
        
        return df
    
    def create_tier2_engineered_aggregates(self, df):
        """Create Tier 2: Engineered lag aggregates"""
        base_series = ['normalized_range', 'bollinger_position', 
                      'volume_ratio', '1d_return', 'vol_ratio', 'vwap_ratio', 'RSI_14']
        windows = [5, 10, 20]
        
        for series in base_series:
            if series in df.columns:  # Only if column exists
                for window in windows:
                    # Rolling statistics
                    df[f'{series}_rolling_mean_{window}'] = df.groupby('Symbol')[series].transform(
                        lambda x: x.rolling(window).mean()
                    )
                    df[f'{series}_rolling_std_{window}'] = df.groupby('Symbol')[series].transform(
                        lambda x: x.rolling(window).std()
                    )
                    df[f'{series}_rolling_max_{window}'] = df.groupby('Symbol')[series].transform(
                        lambda x: x.rolling(window).max()
                    )
                    df[f'{series}_rolling_min_{window}'] = df.groupby('Symbol')[series].transform(
                        lambda x: x.rolling(window).min()
                    )
                    
                    # Rolling trend (slope of linear regression)
                    def rolling_trend(series, window):
                        if len(series) < window:
                            return np.nan
                        x = np.arange(window)
                        y = series.values[-window:]
                        return np.polyfit(x, y, 1)[0]  # Return slope
                    
                    df[f'{series}_rolling_trend_{window}'] = df.groupby('Symbol')[series].transform(
                        lambda x: x.rolling(window).apply(lambda y: rolling_trend(y, window), raw=False)
                    )
        
        return df
    
    def create_tier3_event_memory(self, df):
        """Create Tier 3: Event and regime memory features"""
        # Max drawdown and runup
        df['rolling_max_20'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(20).max())
        df['rolling_min_20'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(20).min())
        
        df['max_drawdown_20'] = np.minimum(0, (df['Close'] / df['rolling_max_20']) - 1)
        df['max_runup_20'] = np.maximum(0, (df['Close'] / df['rolling_min_20']) - 1)
        df['price_position_20d'] = (df['Close'] - df['rolling_min_20']) / (df['rolling_max_20'] - df['rolling_min_20'])
        
        # Volatility regime flags
        if self.fitted_volatility_threshold is None:
            self.fitted_volatility_threshold = df['volatility_20'].quantile(0.75)
        
        df['high_volatility_flag'] = (df['volatility_20'] > self.fitted_volatility_threshold).astype(int)
        df['high_volatility_flag_lag_5'] = df.groupby('Symbol')['high_volatility_flag'].shift(5)
        
        # Volume spike flag
        df['volume_spike_flag'] = (df['volume_ratio'] > 2.0).astype(int)
        
        # Large move flag
        df['large_move_flag'] = (abs(df['1d_return']) > 2 * df['volatility_20']).astype(int)
        
        return df
    
    def create_target_variables(self, df):
        """Create binary target variables: BUY if return > 0, SELL otherwise"""
        # Calculate forward returns
        df['20d_fwd_return'] = (
            df.groupby('Symbol')['Close'].shift(-self.offset_days) / df['Close'] - 1
        )
        
        df['60d_fwd_return'] = (
            df.groupby('Symbol')['Close'].shift(-60) / df['Close'] - 1
        )
        
        # Binary classification: BUY if return > 0, SELL otherwise
        df['20d_signal'] = np.where(df['20d_fwd_return'] > 0, 'BUY', 'SELL')
        df['60d_signal'] = np.where(df['60d_fwd_return'] > 0, 'BUY', 'SELL')
        
        # Also create binary numeric targets for ML (1 for BUY, 0 for SELL)
        df['20d_target'] = (df['20d_fwd_return'] > 0).astype(int)
        df['60d_target'] = (df['60d_fwd_return'] > 0).astype(int)
        
        return df
    
    def clean_dataframe(self, df):
        """Remove unnecessary columns and clean the dataframe"""
        # Remove deliverable-related columns if they exist
        columns_to_remove = ['%Deliverble', 'Deliverable Volume','Series']
        
        # Only remove columns that exist in the dataframe
        columns_to_remove = [col for col in columns_to_remove if col in df.columns]
        df = df.drop(columns=columns_to_remove)
        
        # # Remove intermediate calculation columns
        # intermediate_cols = ['true_range', 'ema_5', 'ema_20', 'rolling_std_5', 'rolling_std_20', 
        #                    'vol_ma_5', 'vol_ma_20', 'day_of_week', 'month', 'day_of_month',
        #                    'rolling_max_20', 'rolling_min_20', 'EMA_12', 'EMA_26', 'SMA_10',
        #                    'SMA_20', 'SMA_50', 'volume_SMA_20', 'vol_ma_5', 'vol_ma_20']
        
        # intermediate_cols = [col for col in intermediate_cols if col in df.columns]
        # df = df.drop(columns=intermediate_cols)
        
        return df
    
    def normalize_features(self, df, fit=False):
        """Normalize numerical features using RobustScaler to handle outliers"""
        feature_cols = [col for col in df.columns]
        
        print(f"Total features before normalization: {feature_cols}")
        
        numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
        
        print(f"Normalizing {numeric_cols} numeric features...")
        
        if fit or self.scaler is None:
            self.scaler = RobustScaler()
            df[numeric_cols] = self.scaler.fit_transform(df[numeric_cols])
        else:
            df[numeric_cols] = self.scaler.transform(df[numeric_cols])
        
        return df

    
    def fit_transform(self, df):
        """Main method to apply all feature engineering"""
        print("Starting comprehensive feature engineering...")
        
        
        # Step 0: Normalize all numeric features
        df = self.normalize_features(df, fit=True)
        print("✓ Features normalized using RobustScaler")
        
        # Step 1: Returns and Moving Averages
        df = self.calculate_returns_and_moving_averages(df)
        print("✓ Returns and moving averages calculated")
        
        # Step 2: Momentum indicators
        df = self.calculate_momentum_indicators(df)
        print("✓ Momentum indicators calculated")
        
        # Step 3: Volatility features
        df = self.calculate_volatility_features(df)
        print("✓ Volatility features calculated")
        
        # Step 4: Volume features
        df = self.calculate_volume_features(df)
        print("✓ Volume features calculated")
        
        # Step 5: Candlestick features
        df = self.calculate_candlestick_features(df)
        print("✓ Candlestick features calculated")
        
        # Step 6: Time features
        df = self.create_time_features(df)
        print("✓ Time features created")
        
        # Step 7: Tier 1 - Raw lags
        df = self.create_tier1_raw_lags(df)
        print("✓ Tier 1 raw lags created")
        
        # Step 8: Tier 2 - Engineered aggregates
        df = self.create_tier2_engineered_aggregates(df)
        print("✓ Tier 2 engineered aggregates created")
        
        # Step 9: Tier 3 - Event memory
        df = self.create_tier3_event_memory(df)
        print("✓ Tier 3 event memory features created")
        
        # Step 10: Target variables
        df = self.create_target_variables(df)
        print("✓ Target variables created")
        
        # Step 11: Clean dataframe
        df = self.clean_dataframe(df)
        print("✓ Dataframe cleaned")
        
        # Remove rows with NaN values
        original_shape = df.shape[0]
        df = df.dropna()
        print(f"✓ Removed rows with NaN. Final dataset: {df.shape[0]} rows (from {original_shape})")
        
        # Print feature information
        feature_columns = [col for col in df.columns if col not in ['Date', 'Symbol', '20d_fwd_return', 
                                                                  '60d_fwd_return', '20d_signal', '60d_signal',
                                                                  '20d_target', '60d_target']]
        print(f"✓ Feature engineering complete! Total features: {len(feature_columns)}")
        
        # Print target distribution
        print(f"\nTarget distribution (20-day):")
        print(df['20d_signal'].value_counts())
        print(f"\nTarget distribution (60-day):")
        print(df['60d_signal'].value_counts())
        
        return df
    
    def get_feature_categories(self, df):
        """Analyze and return feature categories"""
        feature_categories = {
            'price_momentum': [col for col in df.columns if any(x in col for x in [
                'return', 'ROC', 'SMA', 'EMA', 'MACD', 'RSI', 'close_to', 'momentum', 'williams'
            ])],
            'volatility': [col for col in df.columns if any(x in col for x in [
                'volatility', 'range', 'ATR', 'bollinger', 'sharpe', 'std', 'atr'
            ])],
            'volume': [col for col in df.columns if any(x in col for x in [
                'volume', 'OBV', 'vol_', 'vwap'
            ])],
            'candlestick': [col for col in df.columns if any(x in col for x in [
                'candle', 'shadow', 'engulfing'
            ])],
            'time': [col for col in df.columns if any(x in col for x in [
                'dow', 'month', 'is_'
            ])],
            'lags_tier1': [col for col in df.columns if '_lag_' in col],
            'aggregates_tier2': [col for col in df.columns if any(x in col for x in [
                'rolling_mean', 'rolling_std', 'rolling_max', 'rolling_min', 'rolling_trend'
            ])],
            'memory_tier3': [col for col in df.columns if any(x in col for x in [
                'drawdown', 'runup', 'position', 'flag'
            ])]
        }
        
        # Print feature counts
        total_features = 0
        for category, features in feature_categories.items():
            print(f"{category}: {len(features)} features")
            total_features += len(features)
        
        print(f"\nTotal features across categories: {total_features}")
        
        return feature_categories

In [ ]:
def xysplit(df_with_features):
    # Prepare for ML training
    feature_columns = [col for col in df_with_features.columns 
                      if col not in ['Date', 'Symbol', '20d_fwd_return', '60d_fwd_return', 
                                   '20d_signal', '60d_signal', '20d_target', '60d_target']]
    
    X = df_with_features[feature_columns]
    y_20d = df_with_features['20d_target']  # Binary: 1 for BUY, 0 for SELL
    # Or use the signal for interpretation: y_20d_signal = df_with_features['20d_signal']
    
    print(f"Features shape: {X.shape}")
    print(f"Target distribution: {df_with_features['20d_signal'].value_counts()}")
    
    # You can also get the actual return values for analysis
    returns_20d = df_with_features['20d_fwd_return']
    

In [ ]:
def train_test_split(df_with_features,feature_columns):
    # 1. Sort by date first
    df_sorted = df_with_features.sort_values('Date').reset_index(drop=True)

    # 2. Simple time-based split (80% train, 20% test)
    split_point = int(0.80 * len(df_sorted))
    train_df = df_sorted.iloc[:split_point]
    test_df = df_sorted.iloc[split_point:]

    # 3. Prepare features and targets
    X_train = train_df[feature_columns]
    X_test = test_df[feature_columns]
    y_train = train_df['20d_target']
    y_test = test_df['20d_target']

    # 4. Verify no temporal leakage
    print(f"Train period: {train_df['Date'].min()} to {train_df['Date'].max()}")
    print(f"Test period: {test_df['Date'].min()} to {test_df['Date'].max()}")
    assert train_df['Date'].max() <= test_df['Date'].min(), "DATA LEAKAGE DETECTED!"

In [ ]:
df = pd.read_csv('ADANIPORTS_adjusted.csv')
df['Date'] = pd.to_datetime(df['Date'])


In [ ]:
# With your actual data
engineer = FinancialFeatureEngineer(offset_days=20)
df_with_features = engineer.fit_transform(df)

# Get feature categories
categories = engineer.get_feature_categories(df_with_features)
